# **Modelling**

#### Imports

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pydicom
import cv2
from tqdm import tqdm
import torch
from ultralytics import YOLO
from pathlib import Path
import random
import os

#### Paths & Directories

In [ ]:
# Define paths
DATA_DIR = '../data'
TRAIN_IMAGES_DIR = os.path.join(DATA_DIR, 'stage_2_train_images')
TRAIN_LABELS_PATH = os.path.join(DATA_DIR, 'stage_2_train_labels.csv')
CLASS_INFO_PATH = os.path.join(DATA_DIR, 'stage_2_detailed_class_info.csv')
OUTPUT_DIR = '../data/models'
YOLO_DATA_DIR = os.path.join(DATA_DIR, 'yolo_data')

# Create necessary directories
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(YOLO_DATA_DIR, exist_ok=True)

# Create directories with absolute paths to avoid path issues
for split in ['train', 'val', 'test']:
    os.makedirs(os.path.join(os.path.abspath(YOLO_DATA_DIR), 'images', split), exist_ok=True)
    os.makedirs(os.path.join(os.path.abspath(YOLO_DATA_DIR), 'labels', split), exist_ok=True)

### **1. Preprocessing**

In [ ]:
# Function to read DICOM image and preprocess it
def read_dicom(path, size=(640, 640)):
    dicom = pydicom.dcmread(path)
    img = dicom.pixel_array
    orig_height, orig_width = img.shape[:2]
    
    # Convert to float and normalize
    img = img.astype(np.float32) / np.max(img)
    
    # Convert to 3 channels (grayscale to RGB)
    img = np.stack([img, img, img], axis=-1)
    
    # Resize
    img = cv2.resize(img, size)
    
    return img, orig_width, orig_height

# Function to convert bounding box to YOLO format
def convert_to_yolo_format(x, y, width, height, img_width, img_height):
    # YOLO format: class_id, x_center, y_center, width, height (normalized)
    x_center = (x + width / 2) / img_width
    y_center = (y + height / 2) / img_height
    width = width / img_width
    height = height / img_height
    
    return x_center, y_center, width, height

In [ ]:
# Load labels and class info
train_labels = pd.read_csv(TRAIN_LABELS_PATH)
class_info = pd.read_csv(CLASS_INFO_PATH)

# Check what columns are in class_info
print("Columns in train_labels:", train_labels.columns.tolist())
print("\nFirst few rows:")
print(train_labels.head())
print("\nColumns in class_info:", class_info.columns.tolist())
print("\nFirst few rows:")
print(class_info.head())

In [ ]:
# Split data into train, validation, and test sets (70/15/15)
patient_ids = train_labels['patientId'].unique()
random.seed(42)
random.shuffle(patient_ids)

In [ ]:
train_split = int(len(patient_ids) * 0.7)
val_split = int(len(patient_ids) * 0.85)
train_ids = patient_ids[:train_split]
val_ids = patient_ids[train_split:val_split]
test_ids = patient_ids[val_split:]

print(f"Total patients: {len(patient_ids)}")
print(f"Training patients: {len(train_ids)}")
print(f"Validation patients: {len(val_ids)}")
print(f"Test patients: {len(test_ids)}")

In [ ]:
# Create YAML configuration file for YOLOv8 with absolute paths
abs_yolo_data_dir = os.path.abspath(YOLO_DATA_DIR)
yaml_content = f"""
path: {abs_yolo_data_dir}
train: images/train
val: images/val
test: images/test
nc: 1
names: ['pneumonia']
"""

with open(os.path.join(YOLO_DATA_DIR, 'pneumonia.yaml'), 'w') as f:
    f.write(yaml_content)

In [ ]:
bool_convert = False

# Process and convert data to YOLO format
def process_dataset(patient_ids, split):
    # Create directories if they don't exist
    os.makedirs(os.path.join(abs_yolo_data_dir, 'images', split), exist_ok=True)
    os.makedirs(os.path.join(abs_yolo_data_dir, 'labels', split), exist_ok=True)
    
    for patient_id in tqdm(patient_ids, desc=f"Processing {split} set"):
        # Get image path
        image_path = os.path.join(TRAIN_IMAGES_DIR, f"{patient_id}.dcm")
        
        # Define output paths
        output_image_path = os.path.join(abs_yolo_data_dir, 'images', split, f"{patient_id}.png")
        output_label_path = os.path.join(abs_yolo_data_dir, 'labels', split, f"{patient_id}.txt")
        
        # Skip if both files already exist
        if os.path.exists(output_image_path) and os.path.exists(output_label_path):
            continue
        
        # Read and preprocess image
        img, orig_width, orig_height = read_dicom(image_path)
        
        # Save image as PNG for YOLO
        plt.imsave(output_image_path, img)
        
        # Get bounding boxes for this patient
        boxes = train_labels[train_labels['patientId'] == patient_id]
        
        # Create YOLO format label file
        with open(output_label_path, 'w') as f:
            if len(boxes) > 0 and boxes['Target'].sum() > 0:
                # Only process boxes with Target=1 (pneumonia)
                pneumonia_boxes = boxes[boxes['Target'] == 1]
                
                for _, box in pneumonia_boxes.iterrows():
                    x, y, width, height = box['x'], box['y'], box['width'], box['height']
                    x_center, y_center, width, height = convert_to_yolo_format(
                        x, y, width, height, orig_width, orig_height
                    )
                    # Class ID 0 for pneumonia
                    f.write(f"0 {x_center} {y_center} {width} {height}\n")
            else:
                # Create empty file for negative samples
                pass

# Process train, validation, and test sets
if bool_convert:
    print("Converting dataset to YOLO format...")
    process_dataset(train_ids, 'train')
    process_dataset(val_ids, 'val')
    process_dataset(test_ids, 'test')
    print("Dataset conversion complete!")

#### Compare original and pre-processed dataset

In [ ]:
# Compare original DICOM image vs processed YOLO image and their bounding boxes
# Uses the existing notebook variables: TRAIN_IMAGES_DIR, TRAIN_LABELS_PATH, YOLO_DATA_DIR, split
split = 'train'

def yolo_xywh_to_xyxy(yolo_line, img_w, img_h):
    parts = yolo_line.strip().split()
    if len(parts) != 5:
        return None
    _, x_center, y_center, width, height = map(float, parts)
    x1 = (x_center - width / 2) * img_w
    y1 = (y_center - height / 2) * img_h
    x2 = (x_center + width / 2) * img_w
    y2 = (y_center + height / 2) * img_h
    return [max(0, int(x1)), max(0, int(y1)), min(img_w, int(x2)), min(img_h, int(y2))]

def draw_box(img, box, color=(0, 255, 0), thickness=2):
    img = img.copy()
    if img.ndim == 2:
        img = cv2.cvtColor(img, cv2.COLOR_GRAY2RGB)
    x1, y1, x2, y2 = [int(v) for v in box]
    cv2.rectangle(img, (x1, y1), (x2, y2), color, thickness)
    return img

def get_patient_ids_for_split(split_name):
    image_dir = os.path.normpath(os.path.join(YOLO_DATA_DIR, "images", split_name))
    if not os.path.isdir(image_dir):
        return []
    return [f.replace(".png", "") for f in os.listdir(image_dir) if f.endswith(".png")]

def compare_original_vs_processed(patient_id, split_name=split):
    original_path = os.path.normpath(os.path.join(TRAIN_IMAGES_DIR, f"{patient_id}.dcm"))
    if not os.path.exists(original_path):
        raise FileNotFoundError(f"Original DICOM not found: {original_path}")

    dcm = pydicom.dcmread(original_path)
    original_array = dcm.pixel_array.astype(np.float32)
    if original_array.size == 0:
        raise ValueError(f"Empty DICOM pixel array for patient {patient_id}")

    if original_array.max() > 0:
        original_array = original_array / original_array.max()

    orig_h, orig_w = original_array.shape[:2]

    target_size = (640, 640)
    original_rgb = np.stack([original_array, original_array, original_array], axis=-1)
    original_rgb = cv2.resize(original_rgb, target_size, interpolation=cv2.INTER_LINEAR)

    labels_df = pd.read_csv(TRAIN_LABELS_PATH)
    patient_boxes = labels_df[(labels_df["patientId"] == patient_id) & (labels_df["Target"] == 1)].copy()

    original_boxes = []
    for _, row in patient_boxes.iterrows():
        x = float(row["x"])
        y = float(row["y"])
        w = float(row["width"])
        h = float(row["height"])

        x1 = x * (target_size[0] / orig_w)
        y1 = y * (target_size[1] / orig_h)
        x2 = (x + w) * (target_size[0] / orig_w)
        y2 = (y + h) * (target_size[1] / orig_h)

        original_boxes.append([int(x1), int(y1), int(x2), int(y2)])

    original_vis = original_rgb.copy()
    for box in original_boxes:
        original_vis = draw_box(original_vis, box, color=(0, 255, 0), thickness=2)

    processed_path = os.path.normpath(os.path.join(YOLO_DATA_DIR, "images", split_name, f"{patient_id}.png"))
    if not os.path.exists(processed_path):
        raise FileNotFoundError(f"Processed image not found: {processed_path}")

    processed = cv2.imread(processed_path, cv2.IMREAD_COLOR)
    if processed is None:
        raise ValueError(f"Could not read processed image: {processed_path}")

    processed = cv2.cvtColor(processed, cv2.COLOR_BGR2RGB)

    label_path = os.path.normpath(os.path.join(YOLO_DATA_DIR, "labels", split_name, f"{patient_id}.txt"))
    yolo_boxes = []
    if os.path.exists(label_path):
        with open(label_path, "r") as f:
            lines = [line.strip() for line in f if line.strip()]
        for line in lines:
            box = yolo_xywh_to_xyxy(line, processed.shape[1], processed.shape[0])
            if box is not None:
                yolo_boxes.append(box)

    processed_vis = processed.copy()
    for box in yolo_boxes:
        processed_vis = draw_box(processed_vis, box, color=(0, 0, 255), thickness=2)

    print(f"Patient ID: {patient_id}")
    print(f"Split: {split_name}")
    print(f"Original DICOM shape: {orig_h}x{orig_w}")
    print(f"YOLO processed image shape: {processed.shape[0]}x{processed.shape[1]}")

    if patient_boxes.empty:
        print("No positive pneumonia boxes found in the CSV for this patient.")
    else:
        csv_box = patient_boxes.iloc[0][["x", "y", "width", "height"]].to_dict()
        print(f"Original CSV box: {csv_box}")
        print(f"Scaled to 640x640 before processing: {original_boxes[0]}")

    if yolo_boxes:
        print("YOLO normalized label(s):")
        with open(label_path, "r") as f:
            print(f.read().strip())
        print(f"Decoded YOLO boxes on processed image: {yolo_boxes}")
    else:
        print("No YOLO boxes found in label file for this patient.")

    fig, axes = plt.subplots(1, 2, figsize=(18, 8))
    axes[0].imshow(original_vis)
    axes[0].set_title(
        f"Original DICOM (before processing)\n"
        f"Patient: {patient_id} | Original resolution: {orig_w}x{orig_h} | Displayed: {original_vis.shape[1]}x{original_vis.shape[0]}"
    )
    axes[0].axis("off")

    axes[1].imshow(processed_vis)
    axes[1].set_title(
        f"YOLO image (after processing)\n"
        f"Split: {split_name} | Resolution: {processed_vis.shape[1]}x{processed_vis.shape[0]}"
    )
    axes[1].axis("off")

    plt.tight_layout()
    plt.show()

    return {
        "patient_id": patient_id,
        "original_boxes": original_boxes,
        "yolo_boxes": yolo_boxes,
        "original_path": original_path,
        "processed_path": processed_path,
        "label_path": label_path,
    }

processed_patient_ids = get_patient_ids_for_split(split)

if not processed_patient_ids:
    raise ValueError(f"No processed images found in {os.path.join(YOLO_DATA_DIR, 'images', split)}")

labels_df = pd.read_csv(TRAIN_LABELS_PATH)
patients_with_boxes = labels_df[labels_df["Target"] == 1]["patientId"].unique().tolist()
valid_patients = [p for p in processed_patient_ids if p in patients_with_boxes]

if not valid_patients:
    raise ValueError(f"No processed images with bounding boxes found in {split} split")

selected_patient = random.choice(valid_patients)
comparison = compare_original_vs_processed(selected_patient, split_name=split)


### **2. Model training**

In [ ]:
# Initialize and train YOLOv8 model
def train_yolo_model():
    print("Training YOLOv8 model...")

    # Check GPU availability
    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    
    # Initialize model
    model = YOLO('yolov8n.pt')  # Use nano model for faster training
    
    # Train the model
    results = model.train(
        data=os.path.abspath(os.path.join(YOLO_DATA_DIR, 'pneumonia.yaml')),
        epochs=20,
        imgsz=640,
        batch=16,
        name='pneumonia_detection',
        project=OUTPUT_DIR,
        patience=20,  # Early stopping
        save=True
    )
    
    return model, results

In [ ]:
bool_train = False

# Execute function to train model
if bool_train:
    model, training_results = train_yolo_model()

In [ ]:
# Continue training from the last checkpoint
# This will resume training and properly maintain best.pt/last.pt
# - best.pt: Saved automatically when validation metrics improve
# - last.pt: Saved after each epoch (most recent checkpoint)

if bool_train:

    latest_weights = os.path.join(OUTPUT_DIR, 'pneumonia_detection', 'weights', 'last.pt')

    # Check if checkpoint exists
    if os.path.exists(latest_weights):
        print(f"Resuming training from: {latest_weights}")
        model = YOLO(latest_weights)
        
        # Continue training for 5 more epochs
        # YOLO automatically:
        # - Updates best.pt when validation metrics improve
        # - Saves last.pt after each epoch
        # - Maintains training history and logs
        results = model.train(
            data=os.path.abspath(os.path.join(YOLO_DATA_DIR, 'pneumonia.yaml')),
            epochs=50,  # Train for 50 more epochs
            imgsz=640,
            batch=16,
            name='pneumonia_detection',  # Same name to continue in same directory
            project=OUTPUT_DIR,
            patience=20,
            save=True,
            resume=False  # False means continue from weights but reset epoch counter
        )
        
        print("\nTraining completed!")
        print(f"Best model saved at: {os.path.join(OUTPUT_DIR, 'pneumonia_detection', 'weights', 'best.pt')}")
        print(f"Last checkpoint saved at: {os.path.join(OUTPUT_DIR, 'pneumonia_detection', 'weights', 'last.pt')}")
    else:
        print(f"No checkpoint found at {latest_weights}")
        print("Please train the initial model first using the train_yolo_model() function above.")

### **3. Model evaluation**

In [ ]:
# Evaluate model
def evaluate_model(model):
    print("Evaluating model...")
    # Validate on validation set
    val_results = model.val(data=os.path.abspath(os.path.join(YOLO_DATA_DIR, 'pneumonia.yaml')), split='val')
    print(f"Validation mAP50: {val_results.box.map50}")
    print(f"Validation mAP50-95: {val_results.box.map}")
    
    # Test on test set
    test_results = model.val(data=os.path.abspath(os.path.join(YOLO_DATA_DIR, 'pneumonia.yaml')), split='test')
    print(f"Test mAP50: {test_results.box.map50}")
    print(f"Test mAP50-95: {test_results.box.map}")
    
    return val_results, test_results

In [ ]:
# Function to evaluate model using competition metric
def evaluate_competition_metric(model, dataset_split='test'):
    print(f"Evaluating model using competition metric on {dataset_split} set...")
    
    # IoU thresholds as specified in the competition
    iou_thresholds = np.arange(0.4, 0.8, 0.05)  # [0.4, 0.45, 0.5, 0.55, 0.6, 0.65, 0.7, 0.75]
    
    # Get all images in the dataset split
    images_dir = os.path.join(abs_yolo_data_dir, 'images', dataset_split)
    labels_dir = os.path.join(abs_yolo_data_dir, 'labels', dataset_split)
    
    image_files = [f for f in os.listdir(images_dir) if f.endswith('.png')]
    
    # Store average precision for each image
    image_aps = []
    
    for image_file in tqdm(image_files, desc=f"Evaluating {dataset_split} images", mininterval=1.0):
        image_path = os.path.join(images_dir, image_file)
        label_path = os.path.join(labels_dir, image_file.replace('.png', '.txt'))
        
        # Get ground truth boxes
        gt_boxes = []
        if os.path.exists(label_path) and os.path.getsize(label_path) > 0:
            with open(label_path, 'r') as f:
                for line in f:
                    parts = line.strip().split()
                    if len(parts) == 5 and parts[0] == '0':  # Class 0 is pneumonia
                        # Convert YOLO format to [x1, y1, x2, y2]
                        x_center, y_center, width, height = map(float, parts[1:])
                        img = cv2.imread(image_path)
                        img_height, img_width = img.shape[:2]
                        
                        x1 = (x_center - width/2) * img_width
                        y1 = (y_center - height/2) * img_height
                        x2 = (x_center + width/2) * img_width
                        y2 = (y_center + height/2) * img_height
                        
                        gt_boxes.append([x1, y1, x2, y2])
        
        # Get predicted boxes
        results = model.predict(image_path, conf=0.12, max_det=2, verbose=False)
        pred_boxes = []
        confidences = []
        
        if len(results) > 0 and len(results[0].boxes) > 0:
            boxes = results[0].boxes
            for i in range(len(boxes)):
                box = boxes[i].xyxy[0].cpu().numpy()  # [x1, y1, x2, y2]
                conf = boxes[i].conf[0].cpu().numpy()
                pred_boxes.append(box)
                confidences.append(conf)
        
        # Sort predictions by confidence (higher first)
        if pred_boxes:
            sorted_indices = np.argsort(confidences)[::-1]
            pred_boxes = [pred_boxes[i] for i in sorted_indices]

        # No ground truth and no predictions: image is excluded from the metric
        if len(gt_boxes) == 0 and len(pred_boxes) == 0:
            continue
        
        # Calculate precision at each IoU threshold
        precisions = []
        
        for threshold in iou_thresholds:
            # If no ground truth but have predictions, precision is 0
            if len(gt_boxes) == 0 and len(pred_boxes) > 0:
                precisions.append(0.0)
                continue
            
            # Track which gt boxes have been matched
            gt_matched = [False] * len(gt_boxes)
            
            # Count TP, FP, FN
            tp = 0
            fp = 0
            
            for pred_box in pred_boxes:
                # Find best matching ground truth box
                best_iou = 0
                best_gt_idx = -1
                
                for gt_idx, gt_box in enumerate(gt_boxes):
                    if not gt_matched[gt_idx]:
                        iou = calculate_iou(pred_box, gt_box)
                        if iou > best_iou:
                            best_iou = iou
                            best_gt_idx = gt_idx
                
                # Check if the best match exceeds the threshold
                if best_iou > threshold and best_gt_idx != -1:
                    tp += 1
                    gt_matched[best_gt_idx] = True
                else:
                    fp += 1
            
            # Count false negatives (unmatched ground truth boxes)
            fn = sum(1 for matched in gt_matched if not matched)
            
            # Calculate precision
            precision = tp / (tp + fp + fn) if (tp + fp + fn) > 0 else 0
            precisions.append(precision)
        
        # Calculate average precision for this image
        ap = np.mean(precisions)
        image_aps.append(ap)
    
    # Calculate mean average precision across all images
    map_score = np.mean(image_aps)
    
    print(f"Competition mAP score on {dataset_split} set: {map_score:.4f}")
    return map_score

# Function to calculate IoU between two bounding boxes in [x1, y1, x2, y2] format
def calculate_iou(box1, box2):
    # Calculate intersection area
    x_left = max(box1[0], box2[0])
    y_top = max(box1[1], box2[1])
    x_right = min(box1[2], box2[2])
    y_bottom = min(box1[3], box2[3])

    if x_right < x_left or y_bottom < y_top:
        return 0.0

    intersection_area = (x_right - x_left) * (y_bottom - y_top)

    # Calculate union area
    box1_area = (box1[2] - box1[0]) * (box1[3] - box1[1])
    box2_area = (box2[2] - box2[0]) * (box2[3] - box2[1])
    union_area = box1_area + box2_area - intersection_area

    # Calculate IoU
    if union_area > 0:
        return intersection_area / union_area
    else:
        return 0.0


In [ ]:
# Execute function to evaluate model using the official RSNA competition metric
latest_weights = os.path.join(OUTPUT_DIR, 'pneumonia_detection4', 'weights', 'best.pt')
model = YOLO(latest_weights)
competition_score = evaluate_competition_metric(model, dataset_split='test')

In [ ]:
# Execute function to evaluate model
val_results, test_results = evaluate_model(model)

### **4. Making predictions**

In [ ]:
# Visualize predictions
def visualize_predictions(model, num_samples=5):
    print("Visualizing predictions...")
    
    # Randomly select test images
    test_images = os.listdir(os.path.join(abs_yolo_data_dir, 'images', 'test'))
    selected_images = random.sample(test_images, min(num_samples, len(test_images)))
    
    plt.figure(figsize=(20, 20))
    
    for i, img_name in enumerate(selected_images):
        img_path = os.path.join(abs_yolo_data_dir, 'images', 'test', img_name)
        
        # Run prediction
        results = model.predict(img_path, conf=0.12, max_det=2)
        
        # Plot results
        plt.subplot(1, num_samples, i+1)
        plt.imshow(results[0].plot())
        plt.title(f"Patient: {img_name.split('.')[0]}")
        plt.axis('off')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, 'prediction_examples.png'))
    plt.show()

In [ ]:
# Execute function to visualize predictions
visualize_predictions(model)

In [ ]:
# Compare model predictions against ground truth boxes for a random test image
def compare_prediction_vs_ground_truth(model, split_name='test', conf=0.25):
    images_dir = os.path.join(abs_yolo_data_dir, 'images', split_name)
    labels_dir = os.path.join(abs_yolo_data_dir, 'labels', split_name)

    image_files = [f for f in os.listdir(images_dir) if f.endswith('.png')]

    # Only consider images that have a non-empty ground truth label file
    positive_files = [
        f for f in image_files
        if os.path.exists(os.path.join(labels_dir, f.replace('.png', '.txt')))
        and os.path.getsize(os.path.join(labels_dir, f.replace('.png', '.txt'))) > 0
    ]

    if not positive_files:
        raise ValueError(f"No images with ground truth boxes found in {split_name} split")

    img_name = random.choice(positive_files)
    img_path = os.path.join(images_dir, img_name)
    label_path = os.path.join(labels_dir, img_name.replace('.png', '.txt'))

    img = cv2.imread(img_path)
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img_h, img_w = img.shape[:2]

    # Ground truth boxes
    gt_boxes = []
    with open(label_path, 'r') as f:
        for line in f:
            box = yolo_xywh_to_xyxy(line, img_w, img_h)
            if box is not None:
                gt_boxes.append(box)

    # Predicted boxes
    results = model.predict(img_path, conf=conf, verbose=False)
    pred_boxes = []
    confidences = []
    if len(results) > 0 and len(results[0].boxes) > 0:
        for box in results[0].boxes:
            pred_boxes.append(box.xyxy[0].cpu().numpy())
            confidences.append(float(box.conf[0].cpu().numpy()))

    # Draw ground truth (green) and predictions (red)
    vis = img.copy()
    for box in gt_boxes:
        vis = draw_box(vis, box, color=(0, 255, 0), thickness=3)
    for box, c in zip(pred_boxes, confidences):
        vis = draw_box(vis, box, color=(255, 0, 0), thickness=2)
        x1, y1 = int(box[0]), int(box[1])
        cv2.putText(vis, f"{c:.2f}", (x1, max(0, y1 - 5)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 0, 0), 2)

    plt.figure(figsize=(8, 8))
    plt.imshow(vis)
    plt.title(
        f"Patient: {img_name.split('.')[0]}\n"
        f"Ground truth (green): {len(gt_boxes)} | Predictions (red): {len(pred_boxes)}"
    )
    plt.axis('off')
    plt.tight_layout()
    plt.show()

    return {
        'image': img_name,
        'gt_boxes': gt_boxes,
        'pred_boxes': pred_boxes,
        'confidences': confidences,
    }

In [ ]:
# Initialize model
model = YOLO(os.path.join(os.path.abspath(OUTPUT_DIR), 'pneumonia_detection4/weights/best.pt'))  # Use nano model for faster training

# Compare model predictions with ground truth on the test set
comparison_result = compare_prediction_vs_ground_truth(model, split_name='test')

In [ ]:
print("Pneumonia detection model training and evaluation complete!")